# BDA 수강 완료 예측 파이프라인
Target: `completed` (0/1 이진 분류)

## 0. 환경 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

SEED = 42
np.random.seed(SEED)

DATA_PATH = '/content/drive/MyDrive/open'

# 한글 폰트 설정 (Colab)
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fe = fm.FontEntry(fname=font_path, name='NanumGothic')
fm.fontManager.ttflist.insert(0, fe)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

SEED = 42
np.random.seed(SEED)

DATA_PATH = 'content/drive/MyDrive/open'

plt.rcParams['font.family'] = 'NanumGothic'  # 한글 폰트
plt.rcParams['axes.unicode_minus'] = False
print('라이브러리 로드 완료')

## 1. 데이터 로드

In [ ]:
train = pd.read_csv(f'{DATA_PATH}/train.csv')
test  = pd.read_csv(f'{DATA_PATH}/test.csv')
sub   = pd.read_csv(f'{DATA_PATH}/sample_submission.csv')

print(f'train shape : {train.shape}')
print(f'test  shape : {test.shape}')
train.head(3)

## 2. EDA

In [ ]:
# 기본 정보
print('=== 데이터 타입 & 결측치 ===' )
info_df = pd.DataFrame({
    'dtype'   : train.dtypes,
    'null_cnt': train.isnull().sum(),
    'null_pct': (train.isnull().mean() * 100).round(2),
    'nunique' : train.nunique()
})
display(info_df[info_df['null_cnt'] > 0])
print(f'\n총 결측 컬럼 수: {(train.isnull().sum() > 0).sum()}')

In [ ]:
# 타겟 분포
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

vc = train['completed'].value_counts()
axes[0].bar(vc.index.astype(str), vc.values, color=['#5B9BD5','#ED7D31'])
axes[0].set_title('Target 분포 (completed)')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=12)

axes[1].pie(vc.values, labels=['미완료(0)', '완료(1)'], autopct='%1.1f%%',
            colors=['#5B9BD5','#ED7D31'], startangle=90)
axes[1].set_title('Target 비율')

plt.tight_layout()
plt.show()
print(vc)

In [ ]:
# 수치형 피처 분포
num_cols = train.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c not in ['ID', 'completed']]
print(f'수치형 컬럼: {num_cols}')

if num_cols:
    fig, axes = plt.subplots(len(num_cols), 2, figsize=(12, 4 * len(num_cols)))
    if len(num_cols) == 1:
        axes = [axes]
    for i, col in enumerate(num_cols):
        train[col].hist(ax=axes[i][0], bins=30, color='steelblue', edgecolor='white')
        axes[i][0].set_title(f'{col} 분포')
        train.boxplot(column=col, by='completed', ax=axes[i][1])
        axes[i][1].set_title(f'{col} by completed')
    plt.tight_layout()
    plt.show()

In [ ]:
# 카테고리형 피처 top-10 빈도
cat_cols = train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'ID']
print(f'카테고리형 컬럼 수: {len(cat_cols)}')

fig, axes = plt.subplots(len(cat_cols), 1, figsize=(14, 4 * len(cat_cols)))
if len(cat_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, cat_cols):
    vc = train[col].value_counts().head(10)
    ax.barh(vc.index.astype(str), vc.values, color='steelblue')
    ax.set_title(f'{col} (top 10)')
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# 수치형 상관관계
if num_cols:
    corr = train[num_cols + ['completed']].corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
    plt.title('상관관계 히트맵')
    plt.tight_layout()
    plt.show()

## 3. 전처리

In [ ]:
def preprocess(train_df, test_df):
    """train/test 동일하게 전처리 후 반환"""
    drop_cols = ['ID']
    target_col = 'completed'

    y = train_df[target_col].copy()

    tr = train_df.drop(columns=drop_cols + [target_col], errors='ignore').copy()
    te = test_df.drop(columns=drop_cols + [target_col], errors='ignore').copy()

    all_data = pd.concat([tr, te], axis=0, ignore_index=True)

    # bool dtype → int
    for col in all_data.select_dtypes(include='bool').columns:
        all_data[col] = all_data[col].astype(int)

    # 'True'/'False' 문자열 → int
    for col in all_data.select_dtypes(include='object').columns:
        unique_vals = set(all_data[col].dropna().unique())
        if unique_vals <= {'True', 'False'}:
            all_data[col] = all_data[col].map({'True': 1, 'False': 0})

    # 카테고리형 → Label Encoding
    cat_cols = all_data.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        le = LabelEncoder()
        all_data[col] = le.fit_transform(all_data[col].astype(str))

    # 결측치 처리: 전체 NaN 컬럼도 유지 (keep_empty_features=True)
    num_cols = all_data.select_dtypes(include='number').columns.tolist()
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    all_data[num_cols] = imputer.fit_transform(all_data[num_cols])

    X_train = all_data.iloc[:len(tr)].reset_index(drop=True)
    X_test  = all_data.iloc[len(tr):].reset_index(drop=True)

    return X_train, X_test, y, cat_cols

X, X_test, y, cat_features = preprocess(train, test)
print(f'X shape     : {X.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'카테고리 피처 수: {len(cat_features)}')

## 4. Train / Validation 분리 (8:2)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y          # 클래스 비율 유지
)

print(f'X_train : {X_train.shape}  |  y_train 분포: {dict(y_train.value_counts())}')
print(f'X_val   : {X_val.shape}  |  y_val   분포: {dict(y_val.value_counts())}')

## ── 공통 평가 함수 ───────────────────────────────────────────────────────────
def evaluate(name, model, X_tr, y_tr, X_v, y_v, fit=True):
    if fit:
        model.fit(X_tr, y_tr)
    pred = model.predict(X_v)
    prob = model.predict_proba(X_v)[:, 1]
    acc  = accuracy_score(y_v, pred)
    f1   = f1_score(y_v, pred, average='binary')
    auc  = roc_auc_score(y_v, prob)
    return {'name': name, 'model': model, 'prob': prob,
            'acc': acc, 'f1': f1, 'auc': auc}

def show(r):
    print(f"[{r['name']:30s}]  ACC={r['acc']:.4f}  F1={r['f1']:.4f}  AUC={r['auc']:.4f}")

base_results = []   # 단일 모델
ens_results  = []   # 앙상블
all_results  = []   # 전체 비교용

In [ ]:
## ── 5. 베이스 모델 3개 학습 & Val 검증 ─────────────────────────────────────
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    random_state=SEED, verbose=-1)

xgb_model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    eval_metric='logloss', random_state=SEED, verbosity=0)

cat_model = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6,
    random_seed=SEED, verbose=0)

print('=' * 60)
print('[ 단일 모델 Val 성능 ]')
print('=' * 60)
for name, m in [('LightGBM', lgb_model), ('XGBoost', xgb_model), ('CatBoost', cat_model)]:
    r = evaluate(name, m, X_train, y_train, X_val, y_val)
    base_results.append(r)
    all_results.append(r)
    show(r)

model_pool = {r['name']: r for r in base_results}
print()
print('베이스 모델 학습 완료')

## 6. 앙상블 조합 비교 (3가지 조합 × Soft / Weighted)

In [ ]:
from itertools import combinations as combos

COMBOS = [
    ['LightGBM', 'XGBoost'],
    ['LightGBM', 'CatBoost'],
    ['XGBoost',  'CatBoost'],
    ['LightGBM', 'XGBoost', 'CatBoost'],   # 전체 조합
]

print('=' * 60)
print('[ 앙상블 Val 성능 ]')
print('=' * 60)

for names in COMBOS:
    probs = np.column_stack([model_pool[n]['prob'] for n in names])
    aucs  = [roc_auc_score(y_val, probs[:, i]) for i in range(len(names))]
    w     = np.array(aucs) / sum(aucs)
    tag   = '+'.join([n[:3] for n in names])   # e.g. Lig+XGB+Cat

    for method, prob in [('Soft',     probs.mean(axis=1)),
                         ('Weighted', probs @ w)]:
        pred = (prob >= 0.5).astype(int)
        r = {
            'name' : f'{tag} [{method}]',
            'model': None,
            'prob' : prob,
            'acc'  : accuracy_score(y_val, pred),
            'f1'   : f1_score(y_val, pred),
            'auc'  : roc_auc_score(y_val, prob),
            'combo': names,
            'method': method,
        }
        ens_results.append(r)
        all_results.append(r)
        show(r)

In [ ]:
# 전체 결과 테이블 + 시각화
df_all = pd.DataFrame([{k: v for k, v in r.items() if k not in ('model','prob','combo','method')}
                        for r in all_results]).sort_values('auc', ascending=False).reset_index(drop=True)
df_all.index += 1
display(df_all.style.highlight_max(subset=['acc','f1','auc'], color='#c6efce')
                     .format({'acc':'{:.4f}','f1':'{:.4f}','auc':'{:.4f}'}))

# 시각화
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(df_all))
w = 0.28
ax.bar(x - w, df_all['acc'], w, label='ACC',  color='#5B9BD5')
ax.bar(x,     df_all['f1'],  w, label='F1',   color='#ED7D31')
ax.bar(x + w, df_all['auc'], w, label='AUC',  color='#70AD47')
ax.set_xticks(x)
ax.set_xticklabels(df_all['name'], rotation=40, ha='right', fontsize=9)
ax.set_ylim(0.5, 1.05)
ax.axhline(df_all['auc'].max(), color='red', linestyle='--', linewidth=0.8, label=f'Best AUC={df_all["auc"].max():.4f}')
ax.legend()
ax.set_title('모든 모델 Val 성능 비교')
plt.tight_layout()
plt.show()

# 최고 앙상블 저장
best_ens = sorted(ens_results, key=lambda r: r['auc'], reverse=True)[0]
print(f'\n최고 앙상블: {best_ens["name"]}  AUC={best_ens["auc"]:.4f}')

## 7. 하이퍼파라미터 튜닝 (Optuna) — 반복 실행으로 계속 개선

In [ ]:
!pip install -q optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('Optuna 준비 완료')

# ── LightGBM 하이퍼파라미터 튜닝 ─────────────────────────────────────────────
# ★ N_TRIALS 조정으로 탐색 횟수 늘릴 수 있음 (반복 실행 가능)
N_TRIALS = 50

def lgb_objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 16, 128),
        'max_depth'        : trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'random_state'     : SEED,
        'verbose'          : -1,
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X_train, y_train)
    prob = m.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, prob)

lgb_study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n[LightGBM 튜닝 완료]')
print(f'  Best AUC : {lgb_study.best_value:.4f}')
print(f'  Best params: {lgb_study.best_params}')

# 튜닝된 LGB 재학습
lgb_tuned = lgb.LGBMClassifier(**lgb_study.best_params, verbose=-1)
r_lgb_tuned = evaluate('LightGBM [Tuned]', lgb_tuned, X_train, y_train, X_val, y_val)
all_results.append(r_lgb_tuned)
show(r_lgb_tuned)

# ── XGBoost 튜닝 ─────────────────────────────────────────────────────────────
def xgb_objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth'       : trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma'           : trial.suggest_float('gamma', 0, 5),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'eval_metric'     : 'logloss',
        'random_state'    : SEED,
        'verbosity'       : 0,
    }
    m = xgb.XGBClassifier(**params)
    m.fit(X_train, y_train)
    return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])

xgb_study = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

xgb_tuned = xgb.XGBClassifier(**xgb_study.best_params, verbosity=0)
r_xgb_tuned = evaluate('XGBoost [Tuned]', xgb_tuned, X_train, y_train, X_val, y_val)
all_results.append(r_xgb_tuned)
print(f'Best AUC: {xgb_study.best_value:.4f}')
show(r_xgb_tuned)

# ── CatBoost 튜닝 ─────────────────────────────────────────────────────────────
def cat_objective(trial):
    params = {
        'iterations'  : trial.suggest_int('iterations', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth'        : trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg'  : trial.suggest_float('l2_leaf_reg', 1e-4, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_seed'  : SEED,
        'verbose'      : 0,
    }
    m = CatBoostClassifier(**params)
    m.fit(X_train, y_train)
    return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])

cat_study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
cat_study.optimize(cat_objective, n_trials=N_TRIALS, show_progress_bar=True)

cat_tuned = CatBoostClassifier(**cat_study.best_params, verbose=0)
r_cat_tuned = evaluate('CatBoost [Tuned]', cat_tuned, X_train, y_train, X_val, y_val)
all_results.append(r_cat_tuned)
print(f'Best AUC: {cat_study.best_value:.4f}')
show(r_cat_tuned)

In [ ]:
# ── 튜닝된 모델로 앙상블 가중치 최적화 (Optuna) ─────────────────────────────
tuned_pool = {
    'LGB_T': r_lgb_tuned,
    'XGB_T': r_xgb_tuned,
    'CAT_T': r_cat_tuned,
}
tuned_probs = np.column_stack([r['prob'] for r in tuned_pool.values()])

def weight_objective(trial):
    w = np.array([
        trial.suggest_float('w_lgb', 0.1, 0.8),
        trial.suggest_float('w_xgb', 0.1, 0.8),
        trial.suggest_float('w_cat', 0.1, 0.8),
    ])
    w = w / w.sum()   # 합이 1이 되도록 정규화
    prob = tuned_probs @ w
    return roc_auc_score(y_val, prob)

w_study = optuna.create_study(direction='maximize',
                               sampler=optuna.samplers.TPESampler(seed=SEED))
w_study.optimize(weight_objective, n_trials=200, show_progress_bar=True)

best_w = np.array([w_study.best_params['w_lgb'],
                   w_study.best_params['w_xgb'],
                   w_study.best_params['w_cat']])
best_w = best_w / best_w.sum()

opt_prob = tuned_probs @ best_w
opt_pred = (opt_prob >= 0.5).astype(int)
r_opt = {
    'name': 'Tuned Ensemble [Optuna Weights]',
    'model': None, 'prob': opt_prob,
    'acc' : accuracy_score(y_val, opt_pred),
    'f1'  : f1_score(y_val, opt_pred),
    'auc' : roc_auc_score(y_val, opt_prob),
}
all_results.append(r_opt)

print(f'\n최적 가중치: LGB={best_w[0]:.3f} / XGB={best_w[1]:.3f} / CAT={best_w[2]:.3f}')
show(r_opt)

## 8. 최종 결과 비교 (Before → After 튜닝)

In [ ]:
# 전체 결과 최종 테이블
df_final = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('model','prob','combo','method')}
    for r in all_results
]).sort_values('auc', ascending=False).reset_index(drop=True)
df_final.index += 1

display(df_final.style
        .highlight_max(subset=['acc','f1','auc'], color='#c6efce')
        .highlight_min(subset=['auc'], color='#ffc7ce')
        .format({'acc':'{:.4f}','f1':'{:.4f}','auc':'{:.4f}'}))

# Before/After 비교 (베이스 3개 vs 튜닝 3개 vs 최적 앙상블)
compare = {
    'LGB Before' : next(r['auc'] for r in all_results if r['name']=='LightGBM'),
    'XGB Before' : next(r['auc'] for r in all_results if r['name']=='XGBoost'),
    'CAT Before' : next(r['auc'] for r in all_results if r['name']=='CatBoost'),
    'LGB Tuned'  : r_lgb_tuned['auc'],
    'XGB Tuned'  : r_xgb_tuned['auc'],
    'CAT Tuned'  : r_cat_tuned['auc'],
    'Ens Optuna' : r_opt['auc'],
}
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#9DC3E6']*3 + ['#2E75B6']*3 + ['#FF0000']
bars = ax.bar(compare.keys(), compare.values(), color=colors)
ax.set_ylim(min(compare.values()) - 0.05, 1.02)
ax.set_ylabel('AUC')
ax.set_title('Before vs After 하이퍼파라미터 튜닝 (Val AUC)')
for bar, val in zip(bars, compare.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', fontsize=9)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# 최고 모델 상세 분석
best_r = sorted([r for r in all_results if r['model'] is not None],
                key=lambda r: r['auc'], reverse=True)[0]
print(f'최고 단일 모델: {best_r["name"]}  AUC={best_r["auc"]:.4f}')

pred_best = best_r['model'].predict(X_val)
print('\n' + classification_report(y_val, pred_best, target_names=['미완료(0)','완료(1)']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 혼동 행렬
cm = confusion_matrix(y_val, pred_best)
ConfusionMatrixDisplay(cm, display_labels=['미완료','완료']).plot(ax=axes[0], colorbar=False)
axes[0].set_title(f'Confusion Matrix\n{best_r["name"]}')

# LightGBM 피처 중요도
fi = pd.Series(lgb_tuned.feature_importances_, index=X_train.columns).sort_values()
fi.tail(20).plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('LGB [Tuned] Feature Importance Top 20')

plt.tight_layout()
plt.show()

## 9. 최종 예측 & 제출 파일 생성

# 전체 데이터(train+val)로 재학습 → test 예측
lgb_final = lgb.LGBMClassifier(**lgb_study.best_params, verbose=-1)
xgb_final = xgb.XGBClassifier(**xgb_study.best_params, verbosity=0)
cat_final = CatBoostClassifier(**cat_study.best_params, verbose=0)

for m in [lgb_final, xgb_final, cat_final]:
    m.fit(X, y)

test_probs = np.column_stack([
    lgb_final.predict_proba(X_test)[:, 1],
    xgb_final.predict_proba(X_test)[:, 1],
    cat_final.predict_proba(X_test)[:, 1],
])
final_prob = test_probs @ best_w
final_pred = (final_prob >= 0.5).astype(int)

sub['completed'] = final_pred
sub.to_csv(f'{DATA_PATH}/submission.csv', index=False)
print('제출 파일 저장 완료')
print(sub['completed'].value_counts())
sub.head()

In [ ]:
# 최적 모델 혼동 행렬
best_name = results_df.iloc[0]['name']
best_result = next(r for r in results if r['name'] == best_name)
best_model = best_result['model']
print(f'최적 모델: {best_name}')

if best_model is not None:
    pred_best = best_model.predict(X_val)
    print(classification_report(y_val, pred_best))
    cm = confusion_matrix(y_val, pred_best)
    ConfusionMatrixDisplay(cm).plot()
    plt.title(f'Confusion Matrix - {best_name}')
    plt.show()

## 8. 피처 중요도

In [ ]:
# LightGBM 피처 중요도
fi = pd.Series(lgb_model.feature_importances_, index=X_train.columns)
fi = fi.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
fi.sort_values().plot(kind='barh', color='steelblue')
plt.title('LightGBM Feature Importance (Top 20)')
plt.tight_layout()
plt.show()

## 9. 최종 예측 & 제출 파일 생성

In [ ]:
# ── 전체 학습 데이터로 재학습 후 test 예측 ─────────────────
# (Weighted Average 기준, 필요에 따라 best_model.predict 로 교체)

final_models = [
    lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=SEED, verbose=-1),
    xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, use_label_encoder=False,
                      eval_metric='logloss', random_state=SEED, verbosity=0),
    CatBoostClassifier(iterations=500, learning_rate=0.05, random_seed=SEED, verbose=0),
    RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
]

test_probs = []
for m in final_models:
    m.fit(X, y)  # 전체 학습 데이터 사용
    test_probs.append(m.predict_proba(X_test)[:, 1])

# AUC 기반 가중치 재사용
final_prob = np.column_stack(test_probs) @ weights
final_pred = (final_prob >= 0.5).astype(int)

sub['completed'] = final_pred
sub.to_csv(f'{DATA_PATH}/submission.csv', index=False)
print('제출 파일 저장 완료:')
print(sub['completed'].value_counts())
sub.head()